# 01b: 特徴量エンジン検証

実データで特徴量エンジン(`src/features/`)を動作させ、各特徴量の品質を確認する。

In [2]:
import os
os.environ["PGPASSWORD"] = "aa8940aa"

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd

fm.fontManager.addfont(r"C:\Windows\Fonts\YuGothM.ttc")
plt.rcParams["font.family"] = "Yu Gothic"
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 120

from db.connection import DatabaseConnection
from features.feature_engine import FeatureEngine
from features.leakage_validators import validate_no_future_leakage

conn = DatabaseConnection()
engine = conn.get_engine()
print("Setup complete")

Setup complete


In [3]:
print("データをロード中...")
race_df = conn.load_races("20230101", "20261231")
entry_df = conn.load_entries_with_results("20230101", "20261231")
odds_df = conn.load_odds_snapshots("20230101", "20261231")

# 時系列オッズは80M行あるため直近1年分のみ
odds_ts_df = conn.load_odds_time_series_range("20240101", "20261231")

print(f"  races: {len(race_df):,}")
print(f"  entries: {len(entry_df):,}")
print(f"  odds_snapshots: {len(odds_df):,}")
print(f"  odds_timeseries: {len(odds_ts_df):,}")

データをロード中...
  races: 17,808
  entries: 219,004
  odds_snapshots: 149,184
  odds_timeseries: 16,336,438


In [4]:
print("特徴量を生成中...")
feat_engine = FeatureEngine()
feat_df = feat_engine.build_all(race_df, entry_df, odds_df, odds_ts_df=odds_ts_df)
print(f"  生成データ: {len(feat_df):,} 行 × {len(feat_df.columns)} 列")

特徴量を生成中...
  生成データ: 219,004 行 × 53 列


In [5]:
print("=== 欠損値チェック ===")
nulls = feat_df.isnull().sum()
null_cols = nulls[nulls > 0].sort_values(ascending=False)
if len(null_cols) == 0:
    print("  欠損値なし ✓")
else:
    for col, cnt in null_cols.items():
        pct = cnt / len(feat_df) * 100
        print(f"  {col:40s} {cnt:>8,} ({pct:.1f}%)")

=== 欠損値チェック ===
  odds_drop_rate_60_10                      197,180 (90.0%)
  odds_velocity                             116,620 (53.3%)
  odds_drop_rate_30_10                      116,580 (53.2%)
  odds_volatility                           116,580 (53.2%)
  popularity_change_30_10                   116,580 (53.2%)
  place_odds_actual                          70,999 (32.4%)
  tan_odds                                   70,999 (32.4%)
  fuku_odds                                  70,999 (32.4%)
  p_market_win_adj                           70,999 (32.4%)
  zogen_sa                                   16,761 (7.7%)
  ba_taijyu                                     263 (0.1%)
  weight_diff_from_mean                         263 (0.1%)


In [6]:
import numpy as np
print("=== Inf値チェック ===")
numeric_cols = feat_df.select_dtypes(include=[np.number]).columns
inf_cols = []
for col in numeric_cols:
    if np.isinf(feat_df[col]).any():
        inf_cols.append((col, np.isinf(feat_df[col]).sum()))
if not inf_cols:
    print("  Inf値なし ✓")
else:
    for col, cnt in inf_cols:
        print(f"  {col}: {cnt} 件")

=== Inf値チェック ===
  Inf値なし ✓


In [7]:
print("=== 特徴量の基本統計 ===")
feature_cols = [c for c in feat_df.columns if c not in [
    "race_id", "race_date", "umaban", "ketto_num", "year", "month_day",
    "jyo_cd", "kaiji", "nichiji", "race_num", "surface", "surface_key"
]]
feat_df[feature_cols].describe().T

=== 特徴量の基本統計 ===


,count,mean,min,25%,50%,75%,max,std
year_x,219004.0,2024.056561,2023.0,2023.0,2024.0,2025.0,2026.0,0.90889
track_cd,219004.0,20.877824,10.0,17.0,23.0,24.0,24.0,4.409572
distance,219004.0,1558.551401,800.0,1400.0,1500.0,1800.0,3600.0,327.660914
tenko_cd,219004.0,1.530721,0.0,1.0,1.0,2.0,6.0,0.775653
track_condition_code,219004.0,1.539826,1.0,1.0,1.0,2.0,4.0,0.876839
field_size,219004.0,13.235,3.0,11.0,14.0,16.0,29.0,2.954896
race_date_x,219004,2024-07-14 02:20:03.134189568,2023-01-01 00:00:00,2023-10-06 00:00:00,2024-07-04 00:00:00,2025-04-19 00:00:00,2026-03-22 00:00:00,NaN
finish_pos,219004.0,7.094724,1.0,4.0,7.0,10.0,22.0,4.163827
finish_time,219004.0,1425.952476,0.0,1230.0,1360.0,1533.0,3509.0,345.503516
haron_time_l3,219004.0,251.098943,0.0,0.0,354.0,377.0,999.0,174.37663


In [ ]:
# リーク検証はhist系特徴量用（現在のデータにはhist_*が含まれないためスキップ）
print("=== リーク検証 ===")
hist_cols = [c for c in feat_df.columns if c.startswith("hist_")]
if hist_cols:
    print(f"  hist系特徴量: {hist_cols}")
else:
    print("  hist系特徴量なし — リーク検証対象なし ✓")